In [ ]:
import numpy as np
from jaxtyping import Float, Array, Key, Scalar
import jax.numpy as jnp
import jax.random as jr
from jax.numpy.fft import rfftfreq, fftfreq, irfft, rfft, ifft, fft
import matplotlib.pyplot as plt

from jax import config
config.update("jax_enable_x64",True)

from flax import nnx
rngs = nnx.Rngs(42)

YEARSECONDS=365*24*3600
WEEKSECONDS=7*24*3600
YEAROMEGA=2*np.pi/YEARSECONDS
SUNEARTHRADIUS=1.496e11
c=3e8

ARMLENGHT=2.5e9
ARMLENGHTNORMALIZED=ARMLENGHT/SUNEARTHRADIUS
ALPHATRIANGLE=0
BETATRIANGLE=0

NTIMESAMPS = int(WEEKSECONDS/50)
TOBS = WEEKSECONDS
DT = TOBS/NTIMESAMPS
TIMES = jnp.linspace(0,TOBS,NTIMESAMPS,endpoint=False)
CHANNELS = 2


### LISA motion and geometry ###

def sat1_pos(t):
    alp=ALPHATRIANGLE
    bet=BETATRIANGLE
    omS=YEAROMEGA
    Lrs=ARMLENGHTNORMALIZED
    return jnp.stack([
        jnp.cos(omS*t+alp)+Lrs/(4*jnp.sqrt(3)) * ( jnp.cos(2*omS*t+alp+bet) -3*jnp.cos(alp-bet) ),
        jnp.sin(omS*t+alp)+Lrs/(4*jnp.sqrt(3)) * ( jnp.sin(2*omS*t+alp+bet) -3*jnp.sin(alp-bet) ),
        -Lrs/2 * jnp.cos(omS*t+bet)
    ])


def sat2_pos(t):
    alp=ALPHATRIANGLE
    bet=BETATRIANGLE
    omS=YEAROMEGA
    Lrs=ARMLENGHTNORMALIZED
    return jnp.stack([
        jnp.cos(omS*t+alp)+Lrs/(4*jnp.sqrt(3)) * ( jnp.cos(2*omS*t+alp+bet-2*jnp.pi/3) -3*jnp.cos(alp-bet+2*jnp.pi/3) ),
        jnp.sin(omS*t+alp)+Lrs/(4*jnp.sqrt(3)) * ( jnp.sin(2*omS*t+alp+bet-2*jnp.pi/3) -3*jnp.sin(alp-bet+2*jnp.pi/3) ),
        -Lrs/2 * jnp.cos(omS*t+bet-2*jnp.pi/3)
    ])

def sat3_pos(t):
    alp=ALPHATRIANGLE
    bet=BETATRIANGLE
    omS=YEAROMEGA
    Lrs=ARMLENGHTNORMALIZED
    return jnp.stack([
        jnp.cos(omS*t+alp)+Lrs/(4*jnp.sqrt(3)) * ( jnp.cos(2*omS*t+alp+bet+2*jnp.pi/3) -3*jnp.cos(alp-bet-2*jnp.pi/3) ),
        jnp.sin(omS*t+alp)+Lrs/(4*jnp.sqrt(3)) * ( jnp.sin(2*omS*t+alp+bet+2*jnp.pi/3) -3*jnp.sin(alp-bet-2*jnp.pi/3) ),
        -Lrs/2 * jnp.cos(omS*t+bet+2*jnp.pi/3)
    ])

x_all=jnp.zeros((3,len(TIMES),3)) # first index is xyz
x_all = x_all.at[0].set(sat1_pos(TIMES).T)
x_all = x_all.at[1].set(sat2_pos(TIMES).T)
x_all = x_all.at[2].set(sat3_pos(TIMES).T)

d_all=jnp.zeros((3,len(TIMES),3,3)) # first index is xyz
d_all = d_all.at[0].set(
    (x_all[0][:,None]-x_all[1][:,None])*(x_all[0][:,:,None]-x_all[1][:,:,None])
    -(x_all[0][:,None]-x_all[2][:,None])*(x_all[0][:,:,None]-x_all[2][:,:,None])
)/(2*ARMLENGHTNORMALIZED**2)

d_all = d_all.at[1].set(
    (x_all[1][:,None]-x_all[2][:,None])*(x_all[1][:,:,None]-x_all[2][:,:,None])
    -(x_all[1][:,None]-x_all[0][:,None])*(x_all[1][:,:,None]-x_all[0][:,:,None])
)/(2*ARMLENGHTNORMALIZED**2)

d_all = d_all.at[2].set(
    (x_all[2][:,None]-x_all[0][:,None])*(x_all[2][:,:,None]-x_all[0][:,:,None])
    -(x_all[2][:,None]-x_all[1][:,None])*(x_all[2][:,:,None]-x_all[1][:,:,None])
)/(2*ARMLENGHTNORMALIZED**2)

d_A=(2/jnp.sqrt(3))*d_all[0]
d_E=-(2/3)*d_all[0]-(4/3)*d_all[1]


### Polarization tensors ###


def nhat(theta,phi):
    return jnp.stack([jnp.sin(theta)*jnp.cos(phi),jnp.sin(theta)*jnp.sin(phi),jnp.cos(theta)])

def get_theta_phi(vector):
    theta = jnp.arccos(vector[2])
    phi = jnp.arctan2(vector[1], vector[0])
    return theta, phi

def phat(theta,phi):
    return jnp.stack([jnp.sin(phi),-jnp.cos(phi),0*phi])
    
def qhat(theta,phi):
    return jnp.stack([jnp.cos(theta)*jnp.cos(phi),jnp.cos(theta)*jnp.sin(phi),-jnp.sin(theta)])

def tensor_ab(a, b):
    return jnp.einsum('ik, jk -> ijk', a,b)
    
def contractor_ab_a_b(ab, a, b):
    return jnp.einsum('ijk, ik, jk -> k', ab, a, b)

def scal_prod(a,b):
    return jnp.einsum('ij, ij -> j', a,b)

def e_plus(theta,phi):
    toret = tensor_ab(phat(theta,phi),phat(theta,phi))-tensor_ab(qhat(theta,phi),qhat(theta,phi))
    return toret/jnp.sqrt(2) # This agrees with 2201.08782 and 2009.11845 and is different wrt Allen-Ottewill

def e_cross(theta,phi):
    toret = tensor_ab(phat(theta,phi),qhat(theta,phi))+tensor_ab(qhat(theta,phi),phat(theta,phi))
    return toret/jnp.sqrt(2) # This agrees with 2201.08782 and 2009.11845 and is different wrt Allen-Ottewill

In [ ]:
def NtildaE(f):
    L=ARMLENGHT
    Alisa=3.
    Plisa=15.
    
    fstar=1/(2*jnp.pi*L/c)
    toret=( 1/2 * (2+jnp.cos(f/fstar)) * (Plisa/L)**2 * 10**(-24) * (1+(0.002/f)**4)
           +2*(1+jnp.cos(f/fstar)+(jnp.cos(f/fstar))**2 )
            * (Alisa/L)**2 * 10**(-30) * (1+(0.0004/f)**2) * (1+(f/0.008)**4) * (1/(2*jnp.pi*f))**4 )
    return toret
    
def NtildaT(f):
    L=ARMLENGHT
    Alisa=3.
    Plisa=15.
    
    fstar=1/(2*jnp.pi*L/c)
    toret=( (1-jnp.cos(f/fstar)) * (Plisa/L)**2 * 10**(-24) * (1+(0.002/f)**4)
           +2*(1-jnp.cos(f/fstar))**2 * (Alisa/L)**2 * 10**(-30) * (1+(0.0004/f)**2) * (1+(f/0.008)**4) * (1/(2*jnp.pi*f))**4 )
    return toret

def PSD_E(f):
    return jnp.where(f != 0, NtildaE(f), 0.0)

def PSD_T(f):
    return jnp.where(f != 0, NtildaT(f), 0.0)

A, f_ref, alpha = 3.0, 2.0, -0.0
def PSD_test(f):
    # Power-law PSD
    return A * jnp.where(f != 0, (f / f_ref) ** alpha, 0.0)

def noisegen(rng: Key, wPSD):
    T=TOBS
    N=NTIMESAMPS
    dt=DT
    freqs = rfftfreq(N, dt)
    S=wPSD(freqs)
    
    if N % 2 == 0:                               # Nyquist bin if N even
        S  = S.at[-1].set(0.0)
    
    rng_r, rng_i = jr.split(rng, 2)
    xf = jnp.sqrt(S*N)[:, None] * (jr.normal(rng_r, shape=(N//2+1,CHANNELS))
                                   + 1j * jr.normal(rng_i, shape=(N//2+1,CHANNELS)) )/jnp.sqrt(2.0)

    return irfft(xf,axis=0), S, freqs


xt,Sf,fqs=noisegen(rngs.eval(),PSD_E)

plt.loglog(fqs,jnp.abs(rfft(xt,axis=0))/jnp.sqrt(NTIMESAMPS))
plt.loglog(fqs,jnp.sqrt(PSD_E(fqs)))
plt.xlim(1/TOBS,0.49/DT)
plt.ylim(1e-23,1e-13)
plt.grid()
plt.show()

In [ ]:
### Waveform ###

Parameters = Float[Array, "sources 4"]
CHANNELS = 2
Observation = Float[Array, "times 2"]

SOURCES = 3
AMPLITUDE_RANGE = (1e-21, 1e-20)
CIOTA_RANGE = (-1,1)
F0_RANGE = (1e-4, 1e-2)
PHI0_RANGE = (0, 2 * np.pi)
THETA_RANGE = (0, np.pi)
PHI_RANGE = (0, 2 * np.pi)


def sample_joint(rng: Key) -> tuple[Parameters, Observation]:
    rng_A, rng_ciota, rng_f0, rng_phi0, rng_theta, rng_phi, rng_noise = jr.split(rng, 7)
    log_A = jr.uniform(
        rng_A,
        shape=(SOURCES,),
        minval=np.log(AMPLITUDE_RANGE[0]),
        maxval=np.log(AMPLITUDE_RANGE[1]),
    )
    ciota = jr.uniform(
        rng_ciota,
        shape=(SOURCES,),
        minval=CIOTA_RANGE[0],
        maxval=CIOTA_RANGE[1],
    )
    log_f0 = jr.uniform(
        rng_f0,
        shape=(SOURCES,),
        minval=np.log(F0_RANGE[0]),
        maxval=np.log(F0_RANGE[1]),
    )
    phi0 = jr.uniform(
        rng_phi0,
        shape=(SOURCES,),
        minval=PHI0_RANGE[0],
        maxval=PHI0_RANGE[1],
    )
    
    theta = jr.uniform(
        rng_phi0,
        shape=(SOURCES,),
        minval=THETA_RANGE[0],
        maxval=THETA_RANGE[1],
    )

    phi = jr.uniform(
        rng_phi0,
        shape=(SOURCES,),
        minval=PHI_RANGE[0],
        maxval=PHI_RANGE[1],
    )
    
    A = jnp.exp(log_A)
    f0 = jnp.exp(log_f0)
    x = jnp.stack([A, ciota, f0, phi0, theta, phi], axis=-1)

    
    Phi_GB = 2*jnp.pi*f0*TIMES[..., None]-phi0
    h_plus = A[None]*(1+ciota[None]**2)*jnp.cos(Phi_GB)
    h_cross = 2*A[None]*ciota[None]*jnp.sin(Phi_GB)

    F_plus_A = jnp.einsum('tij, ijs -> ts', d_A, e_plus(theta,phi))
    F_cross_A = jnp.einsum('tij, ijs -> ts', d_A, e_cross(theta,phi))
    
    F_plus_E = jnp.einsum('tij, ijs -> ts', d_E, e_plus(theta,phi))
    F_cross_E = jnp.einsum('tij, ijs -> ts', d_E, e_cross(theta,phi))

    h_A = (h_plus*F_plus_A+h_cross*F_cross_A).sum(-1)
    h_E = (h_plus*F_plus_E+h_cross*F_cross_E).sum(-1)
    
    h = jnp.stack([h_A, h_E], axis=-1)
    n,Sf,fqs = noisegen(rng_noise,PSD_E)
    y = h + n
    return x, y, fqs

In [ ]:
x, y, fqs = sample_joint(rngs.eval())
plt.loglog(fqs,jnp.abs(rfft(y,axis=0)))